# Download historical equity data for NASDAQ stocks from yahoo finance

In [1]:
import warnings

warnings.filterwarnings("ignore")

In [3]:
from time import time
from tqdm import tqdm
from pathlib import Path
import pandas as pd
import yfinance as yf
import requests
from io import StringIO

In [4]:
idx = pd.IndexSlice

In [5]:
results_path = Path("results", "asset_pricing")
if not results_path.exists():
    results_path.mkdir(parents=True)

In [6]:
def chunks(l, n):
    for i in range(0, len(l), n):
        yield l[i : i + n]

In [7]:
def format_time(t):
    """Return a formatted time string 'HH:MM:SS
    based on a numeric time() value"""
    m, s = divmod(t, 60)
    h, m = divmod(m, 60)
    return f"{h:0>2.0f}:{m:0>2.0f}:{s:0>2.0f}"

## Get NASDAQ symbols

In [9]:
def get_nasdaq_symbols():
    url = "https://www.nasdaqtrader.com/dynamic/SymDir/nasdaqlisted.txt"
    response = requests.get(url)
    if response.status_code != 200:
        raise Exception(f"Failed to fetch data: {response.status_code}")

    # Remove the footer line "File Creation Time: ..."
    lines = response.text.strip().split("\n")
    cleaned_text = "\n".join(lines[:-1])

    df = pd.read_csv(StringIO(cleaned_text), sep="|")
    
    # Optional: filter out test issues or other flags
    df = df[df["Test Issue"] == "N"].dropna()
    
    return df

In [10]:
traded_symbols = get_nasdaq_symbols()

In [11]:
traded_symbols.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5000 entries, 0 to 5008
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Symbol            5000 non-null   object
 1   Security Name     5000 non-null   object
 2   Market Category   5000 non-null   object
 3   Test Issue        5000 non-null   object
 4   Financial Status  5000 non-null   object
 5   Round Lot Size    5000 non-null   int64 
 6   ETF               5000 non-null   object
 7   NextShares        5000 non-null   object
dtypes: int64(1), object(7)
memory usage: 351.6+ KB


## Download metadata from yahoo finance

### NASDAQ symbols

In [12]:
all_symbols = list(traded_symbols[traded_symbols.ETF == 'N'].Symbol.unique())
n = len(all_symbols)
print(f"# Symbols: {n:,.0f}")

# Symbols: 4,064


In [13]:
DATA_STORE = '../data/assets.h5'

In [15]:
with pd.HDFStore(DATA_STORE) as store:
    prices = store['/stooq/us/nasdaq/stocks/prices']

In [24]:
with pd.HDFStore(DATA_STORE) as store:
    quandl = store['quandl/wiki/prices']

In [27]:
quandl_filt = quandl[quandl.index.get_level_values(1).isin(all_symbols)]

In [33]:
quandl_filt = quandl_filt[~quandl_filt.index.get_level_values(1).isin(curr_tickers)]

In [35]:
len(quandl_filt.index.get_level_values(1).unique())

48

In [37]:
quandl_final = quandl_filt.loc[idx['2008':'2019', :], :].filter(like='adj_').dropna().swaplevel().rename(columns=lambda x: x.replace('adj_', ''))

In [43]:
prices_filt = prices_filt.dropna()

In [44]:
prices_filt.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 4431762 entries, ('AACG', Timestamp('2008-01-28 00:00:00')) to ('ZYXI', Timestamp('2019-12-31 00:00:00'))
Data columns (total 5 columns):
 #   Column  Dtype  
---  ------  -----  
 0   open    float64
 1   high    float64
 2   low     float64
 3   close   float64
 4   volume  float64
dtypes: float64(5)
memory usage: 186.2+ MB


In [42]:
quandl_final.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 99429 entries, ('ACFN', Timestamp('2008-01-02 00:00:00')) to ('TCBI', Timestamp('2018-03-27 00:00:00'))
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   open    99429 non-null  float64
 1   high    99429 non-null  float64
 2   low     99429 non-null  float64
 3   close   99429 non-null  float64
 4   volume  99429 non-null  float64
dtypes: float64(5)
memory usage: 4.9+ MB


In [47]:
data = pd.concat([prices_filt, quandl_final], axis = 0)

In [49]:
data = data.sort_index(level=['ticker', 'date'])

In [50]:
data

open    high     low   close      volume
ticker date                                                  
AACG   2008-01-28  0.7380  0.7380  0.7380  0.7380         0.0
       2008-01-29  0.7380  0.7760  0.6657  0.6797  19169939.0
       2008-01-30  0.6797  0.7108  0.6448  0.6603   2818723.0
       2008-01-31  0.6595  0.8001  0.6595  0.7418   2345961.0
       2008-02-01  0.7714  0.7721  0.7380  0.7387    361853.0
...                   ...     ...     ...     ...         ...
ZYXI   2019-12-24  7.0929  7.3630  6.9854  7.3360    148995.0
       2019-12-26  7.2999  7.4350  7.1199  7.2370    141086.0
       2019-12-27  7.2189  7.5520  7.1650  7.2460    263370.0
       2019-12-30  7.2460  7.3089  6.9760  7.1650    164479.0
       2019-12-31  7.0570  7.2640  6.9309  7.0839    130535.0

[4531191 rows x 5 columns]

In [18]:
prices.index = prices.index.set_levels(
    prices.index.levels[0].str.replace('.US', '', regex=False), level=0
)

In [19]:
prices

open    high     low   close      volume
ticker date                                                  
AACG   2008-01-28  0.7380  0.7380  0.7380  0.7380         0.0
       2008-01-29  0.7380  0.7760  0.6657  0.6797  19169939.0
       2008-01-30  0.6797  0.7108  0.6448  0.6603   2818723.0
       2008-01-31  0.6595  0.8001  0.6595  0.7418   2345961.0
       2008-02-01  0.7714  0.7721  0.7380  0.7387    361853.0
...                   ...     ...     ...     ...         ...
ZYXI   2019-12-24  7.0929  7.3630  6.9854  7.3360    148995.0
       2019-12-26  7.2999  7.4350  7.1199  7.2370    141086.0
       2019-12-27  7.2189  7.5520  7.1650  7.2460    263370.0
       2019-12-30  7.2460  7.3089  6.9760  7.1650    164479.0
       2019-12-31  7.0570  7.2640  6.9309  7.0839    130535.0

[4654102 rows x 5 columns]

In [20]:
prices_filt = prices[prices.index.get_level_values(0).isin(all_symbols)]

In [31]:
curr_tickers = list(prices_filt.index.get_level_values(0).unique())

In [59]:
yf_symbols = yf.Tickers(all_symbols)

In [60]:
meta_data = []
start = time()
for ticker, yf_object in tqdm(yf_symbols.tickers.items()):
    try:
        s = pd.Series(yf_object.get_info())
        meta_data.append(s.to_frame(ticker))
    except Exception as e:
        # track errors
        print(ticker, e)

print(f"Success: {len(meta_data):5,.0f} / {n:5,.0f}")

100%|██████████| 4064/4064 [17:28<00:00,  3.88it/s]

Success: 4,064 / 4,064


In [61]:
df = pd.concat(meta_data, axis=1).dropna(how="all").T
df = df.apply(pd.to_numeric, errors="ignore")
df.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
Index: 4064 entries, AACB to ZYXI
Columns: 205 entries, address1 to exchangeTransferDate
dtypes: bool(5), float64(140), int64(8), object(52)
memory usage: 6.3+ MB


In [62]:
df.to_hdf(results_path / "data.h5", "stocks/info")

## Download adjusted price data using yfinance

In [73]:
prices_adj = []
start = time()
for i, chunk in enumerate(chunks(all_symbols, 100), 1):
    prices_adj.append(yf.download(chunk, start='1990-01-01', end='2019-12-31', auto_adjust=True).stack(-1))

    per_ticker = (time() - start) / (i * 100)
    to_do = n - (i * 100)
    to_go = to_do * per_ticker
    print(
        f"Success: {len(prices_adj):5,.0f}/{i:5,.0f} | To go: {format_time(to_go)} ({to_do:5,.0f})"
    )

[*********************100%***********************]  100 of 100 completed

51 Failed downloads:
['AACIW', 'ADNWW', 'ABLVW', 'ABPWW', 'ADSEW', 'AEVAW', 'ADVWW', 'ACONW', 'ABLLW', 'ABVEW', 'AERTW', 'AENTW', 'AACBR']: YFPricesMissingError('possibly delisted; no price data found  (1d 1990-01-01 -> 2019-12-31)')
['ADV', 'ADXN', 'AACIU', 'ACTU', 'ABLLL', 'ACOG', 'AAPG', 'ABL', 'ACGLN', 'ACT', 'ABLV', 'ABVE', 'ACRV', 'AACB', 'ADSE', 'ABOS', 'ACXP', 'ADGM', 'AEVA', 'AENT', 'ABNB', 'ABCL', 'ACDC', 'AEBI', 'AEI', 'AERT', 'ABVX', 'ADAG', 'ADVB', 'AACBU', 'ABSI', 'ACON', 'ACLX', 'ADUR', 'ADTX', 'AARD', 'AACI', 'ABP']: YFPricesMissingError('possibly delisted; no price data found  (1d 1990-01-01 -> 2019-12-31) (Yahoo error = "Data doesn\'t exist for startDate = 631170000, endDate = 1577768400")')
[**                     5%                       ]  5 of 100 completed

Success:     1/    1 | To go: 00:06:33 (3,964)


[*********************100%***********************]  100 of 100 completed

50 Failed downloads:
['AIFER', 'ALCYW', 'AIMDW', 'ALFUW', 'ALDFW', 'AISPW', 'AFJKR', 'AIRJW', 'AFRIW', 'ALVOW']: YFPricesMissingError('possibly delisted; no price data found  (1d 1990-01-01 -> 2019-12-31)')
['AIP', 'AFCG', 'AMIX', 'ALF', 'ALVO', 'AGH', 'AFJK', 'ALHC', 'AIRJ', 'ALDF', 'ALCY', 'ALGS', 'AIXI', 'ALZN', 'AGNCL', 'AIRS', 'AIFEU', 'AIFE', 'AGRI', 'AKAN', 'ALXO', 'AFRI', 'ALKT', 'AGFY', 'ALCYU', 'AFJKU', 'ALAB', 'AFRM', 'ALLR', 'AISP', 'ALFUU', 'ALDFU', 'ALTI', 'AIRO', 'ALMS', 'AIMD', 'ALGM', 'AGNCP', 'ALMU', 'AIRE']: YFPricesMissingError('possibly delisted; no price data found  (1d 1990-01-01 -> 2019-12-31) (Yahoo error = "Data doesn\'t exist for startDate = 631170000, endDate = 1577768400")')
[*                      2%                       ]  2 of 100 completed

Success:     2/    2 | To go: 00:05:28 (3,864)


[*********************100%***********************]  100 of 100 completed

46 Failed downloads:
['ASBPW', 'ANGHW', 'ANNAW', 'ARBEW', 'AREBW', 'ARKOW', 'AMODW', 'ANSCW']: YFPricesMissingError('possibly delisted; no price data found  (1d 1990-01-01 -> 2019-12-31)')
['AMST', 'ARM', 'AREB', 'AMPGW', 'AMPG', 'ARBK', 'ANTX', 'ARHS', 'APADU', 'ARBB', 'ANPA', 'ARBE', 'ANSCU', 'ASBP', 'ANNA', 'APLD', 'ANTA', 'AMOD', 'AMLX', 'ARAI', 'ANGH', 'AMPL', 'API', 'ARQT', 'ANL', 'ARBKL', 'APGE', 'ARTV', 'APLM', 'AOUT', 'ANEB', 'ANNX', 'ARQQ', 'ARRY', 'ARQQW', 'APP', 'APLMW', 'ANSC']: YFPricesMissingError('possibly delisted; no price data found  (1d 1990-01-01 -> 2019-12-31) (Yahoo error = "Data doesn\'t exist for startDate = 631170000, endDate = 1577768400")')
[**                     4%                       ]  4 of 100 completed

...

Success:    39/   39 | To go: 00:00:11 (  164)


[*********************100%***********************]  100 of 100 completed

46 Failed downloads:
['WMG', 'XRTX', 'XCH', 'WFRD', 'WW', 'WETO', 'WLDS', 'WTGUU', 'XBP', 'WSBK', 'XAGE', 'XMTR', 'WIMI', 'XOS', 'WTFCN', 'WLAC', 'WSBCP', 'XHLD', 'WNW', 'WFF', 'WGRX', 'XLO', 'XOMAP', 'WRD', 'WYHG', 'WTF', 'WLGS', 'WLACU', 'WGS', 'WOK', 'WHFCL', 'XOMAO', 'XPON', 'WHLRL', 'WOOF', 'WTG', 'WTO', 'WETH', 'WXM']: YFPricesMissingError('possibly delisted; no price data found  (1d 1990-01-01 -> 2019-12-31) (Yahoo error = "Data doesn\'t exist for startDate = 631170000, endDate = 1577768400")')
['XBPEW', 'WLACW', 'WTGUR', 'XOSWW', 'WGSWW', 'XAGEW', 'WLDSW']: YFPricesMissingError('possibly delisted; no price data found  (1d 1990-01-01 -> 2019-12-31)')
[****                   8%                       ]  5 of 64 completed

Success:    40/   40 | To go: 00:00:04 (   64)


[*********************100%***********************]  62 of 64 completed

36 Failed downloads:
['ZOOZW', 'YORKW', 'ZEOWW', 'YHNAR']: YFPricesMissingError('possibly delisted; no price data found  (1d 1990-01-01 -> 2019-12-31)')
['ZJK', 'ZCMD', 'YOSH', 'YOUL', 'YYGH', 'ZENA', 'ZDAI', 'YB', 'YMAT', 'ZIMV', 'ZJYL', 'ZURA', 'YGMZ', 'ZNTL', 'YXT', 'YORKU', 'YHNAU', 'ZSPC', 'ZBAO', 'YHNA', 'ZOOZ', 'YQ', 'YHC', 'YAAS', 'YYAI', 'ZENV', 'ZYBT', 'YIBO', 'ZEO', 'YORK', 'ZBIO', 'YSXT']: YFPricesMissingError('possibly delisted; no price data found  (1d 1990-01-01 -> 2019-12-31) (Yahoo error = "Data doesn\'t exist for startDate = 631170000, endDate = 1577768400")')


Success:    41/   41 | To go: -1:59:58 (  -36)


In [74]:
prices_adj = (
    pd.concat(prices_adj)
    .dropna(how="all", axis=1)
    .rename(columns=str.lower)
    .swaplevel()
)

In [75]:
prices_adj.index.names = ["ticker", "date"]

In [76]:
len(prices_adj.index.unique("ticker"))

2077

### Remove outliers

In [77]:
df = prices_adj.close.unstack("ticker")
pmax = df.pct_change().max()
pmin = df.pct_change().min()
to_drop = pmax[pmax > 1].index.union(pmin[pmin < -1].index)
len(to_drop)

375

In [78]:
prices_adj = prices_adj.drop(to_drop, level="ticker")

In [79]:
len(prices_adj.index.unique("ticker"))

1702

In [80]:
prices_adj

,Price,close,high,low,open,volume
ticker,date,,,,,
AAME,1990-01-02,2.206222,2.206222,1.985599,2.206222,8200.0
AAPL,1990-01-02,0.261499,0.263253,0.245704,0.247458,183198400.0
ACNT,1990-01-02,2.722866,2.722866,2.689660,2.689660,6750.0
ADBE,1990-01-02,1.188340,1.203010,1.144327,1.188340,7166400.0
ADI,1990-01-02,0.990637,1.016368,0.964906,1.016368,895800.0
...,...,...,...,...,...,...
ZTEK,2019-12-30,0.270000,0.280000,0.270000,0.280000,11500.0
ZUMZ,2019-12-30,33.959999,34.189999,32.330002,32.790001,472200.0
ZVRA,2019-12-30,6.720000,8.000000,6.688000,7.360000,194806.0


In [81]:
prices_adj.sort_index().loc[idx[:, '1990': '2019'], :]

Price                  close       high        low       open     volume
ticker date                                                             
AAL    2005-09-27  18.194912  20.174670  18.006365  19.844710   961200.0
       2005-09-28  19.326202  19.354485  18.100639  18.194912  5747900.0
       2005-09-29  19.052803  19.401618  18.949103  19.231924  1078200.0
       2005-09-30  19.806997  19.844706  19.024522  19.099941  3123300.0
       2005-10-03  20.268942  20.504627  19.703297  19.703297  1057900.0
...                      ...        ...        ...        ...        ...
ZYXI   2019-12-23   7.174984   7.327835   6.788362   6.860292   374990.0
       2019-12-24   7.327835   7.354809   6.977177   7.085072   147510.0
       2019-12-26   7.228932   7.426738   7.112046   7.291870   139700.0
       2019-12-27   7.237923   7.543624   7.157002   7.210949   260810.0
       2019-12-30   7.157002   7.300861   6.968187   7.237923   162910.0

[6078122 rows x 5 columns]

In [82]:
prices_adj.sort_index().loc[idx[:, '1990': '2019'], :].to_hdf(results_path / 'data.h5', 
                                                              'stocks/prices/adjusted')

In [54]:
data_final = data.sort_index().loc[idx[:, '2009':'2018'], :]

In [56]:
data_final.describe()

,open,high,low,close,volume
count,3.124659e+06,3.124659e+06,3.124659e+06,3.124659e+06,3.124659e+06
mean,1.255574e+09,1.352508e+09,1.131540e+09,1.271283e+09,2.193442e+06
std,1.207759e+11,1.326576e+11,9.464343e+10,1.206310e+11,2.541785e+07
min,4.200000e-03,4.600000e-03,1.400000e-03,1.500000e-03,0.000000e+00
25%,7.990000e+00,8.135000e+00,7.830000e+00,7.990000e+00,1.554700e+04
50%,1.690900e+01,1.720000e+01,1.660730e+01,1.691000e+01,1.082695e+05
75%,3.788640e+01,3.847890e+01,3.726665e+01,3.788590e+01,5.528005e+05
max,6.135446e+13,6.135446e+13,4.252298e+13,6.074696e+13,3.807199e+09


In [57]:
data_final.isnull().sum()

open      0
high      0
low       0
close     0
volume    0
dtype: int64

In [58]:
data_final.to_hdf(
    results_path / "data.h5", "stocks/prices/adjusted"
)